In [1]:
import pandas as pd
import numpy as np
import itertools
import datetime
import pandas_gbq
import matplotlib.pyplot as plt
from datetime import *
from datetime import datetime, timedelta, date
# %load_ext google.colab.data_table
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
project_id = "perceptive-ivy-290216"

# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
query2=f"""
SELECT * FROM `perceptive-ivy-290216.f1_api.qualifying_lap_time`
# WHERE YEAR=2023
# AND GP='Spanish Grand Prix'
"""
track3=pandas_gbq.read_gbq(query2,project_id,dialect='standard')

Downloading: 100%|██████████|


In [118]:
track2=track3[(track3["Team"].isin(['AlphaTauri','RB','Racing Bulls']))&(track3["Year"].isin([2021,2022,2023,2024,2025]))]

In [119]:
Drivers=(track2['Driver'].unique())
Drivers

array(['DEV', 'GAS', 'LAW', 'RIC', 'TSU', 'HAD'], dtype=object)

In [120]:
#Assign Rank for each entry point
track2["RK"] = track2.groupby(by=["Driver","GP","Year"])["LapTime"].rank(method="dense", ascending=True)
track2['LapTime_TD']= pd.to_timedelta(track2["LapTime"])
track2.loc[:, "LapTime (s)"] = track2["LapTime_TD"].dt.total_seconds()
track2["RK"]= track2["RK"].astype(int)

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_30952/1484189848.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2["RK"] = track2.groupby(by=["Driver","GP","Year"])["LapTime"].rank(method="dense", ascending=True)
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_30952/1484189848.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track2['LapTime_TD']= pd.to_timedelta(track2["LapTime"])
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_30952/1484189848.py:4: SettingWithC

In [121]:
track_fastest=track2[track2['RK']==1]

In [122]:
pole_lap = track_fastest["LapTime"].min()
track_fastest['LapTime_TD']= pd.to_timedelta(track_fastest["LapTime"])
track_fastest["LapTime"]=track_fastest['LapTime'].str.split('days ').str[1]
track_fastest.loc[:, "LapTime (s)"] = track_fastest["LapTime_TD"].dt.total_seconds()
track_fastest['LapTimeDelta']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
track_fastest['LapTimeDelta']=track_fastest['LapTimeDelta'].astype(str)

for index, row in track_fastest.iterrows():
  if track_fastest.loc[index, 'LapTimeDelta']=='0 days 00:00:00':
    track_fastest.loc[index, 'LapTimeDelta']='0 days 00:00:00.01'
track_fastest['LapTimeDelta']=pd.to_timedelta(track_fastest['LapTimeDelta'])
track_fastest

track_fastest['LapTimeDelta'] = track_fastest['LapTimeDelta'] + pd.to_datetime('1970/01/01')

track_fastest['LapTimeDelta2']= pd.to_timedelta(track_fastest["LapTime"]) - pd.to_timedelta(pole_lap)
track_fastest['LapTimeDelta2']=track_fastest['LapTimeDelta2'].astype(str)

track_fastest["LapTimeDelta2"]=track_fastest['LapTimeDelta2'].str.split('days ').str[1]

/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_30952/4155055247.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track_fastest['LapTime_TD']= pd.to_timedelta(track_fastest["LapTime"])
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_30952/4155055247.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  track_fastest["LapTime"]=track_fastest['LapTime'].str.split('days ').str[1]
/var/folders/x_/b65sxrpx6737wtqnt0ctv71w0000gn/T/ipykernel_30952/4155055247.py:5: SettingWithCopyWarning: 
A 

In [123]:
track_fastest.head()

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,Year,GP,RK,LapTime_TD,LapTime (s),LapTimeDelta,LapTimeDelta2
5263,0 days 00:31:26.629000,DEV,21,00:01:32.121000,5.0,2.0,NaT,NaT,0 days 00:00:29.382000,0 days 00:00:39.634000,0 days 00:00:23.105000,0 days 00:30:23.890000,0 days 00:31:03.524000,0 days 00:31:26.629000,240.0,270.0,285.0,315.0,True,SOFT,2.0,True,AlphaTauri,0 days 00:29:54.508000,2023-03-04 15:14:55.505,1,NaN,False,,False,True,2023,Bahrain Grand Prix,1,0 days 00:01:32.121000,92.121,1970-01-01 00:00:28.014,00:00:28.014000
5270,0 days 01:00:13.568000,DEV,21,00:01:18.335000,20.0,4.0,NaT,NaT,0 days 00:00:27.215000,0 days 00:00:17.843000,0 days 00:00:33.277000,0 days 00:59:22.448000,0 days 00:59:40.291000,0 days 01:00:13.568000,282.0,314.0,301.0,318.0,True,SOFT,2.0,True,AlphaTauri,0 days 00:58:55.233000,2023-04-01 05:44:22.283,1,NaN,False,,False,True,2023,Australian Grand Prix,1,0 days 00:01:18.335000,78.335,1970-01-01 00:00:14.228,00:00:14.228000
5271,0 days 00:21:26.236000,DEV,21,00:01:55.282000,2.0,1.0,NaT,NaT,0 days 00:00:41.203000,0 days 00:00:47.497000,0 days 00:00:26.582000,0 days 00:20:12.157000,0 days 00:20:59.654000,0 days 00:21:26.236000,179.0,192.0,330.0,301.0,True,SOFT,2.0,True,AlphaTauri,0 days 00:19:30.954000,2023-04-28 13:05:04.376,1,NaN,False,,False,True,2023,Azerbaijan Grand Prix,1,0 days 00:01:55.282000,115.282,1970-01-01 00:00:51.175,00:00:51.175000
5273,0 days 00:29:45.872000,DEV,21,00:01:28.325000,7.0,2.0,NaT,NaT,0 days 00:00:29.529000,0 days 00:00:33.683000,0 days 00:00:25.113000,0 days 00:28:47.076000,0 days 00:29:20.759000,0 days 00:29:45.872000,223.0,194.0,281.0,332.0,True,SOFT,2.0,True,AlphaTauri,0 days 00:28:17.547000,2023-05-06 20:13:18.533,1,NaN,False,,False,True,2023,Miami Grand Prix,1,0 days 00:01:28.325000,88.325,1970-01-01 00:00:24.218,00:00:24.218000
5279,0 days 01:03:55.875000,DEV,21,00:01:12.428000,19.0,5.0,NaT,NaT,0 days 00:00:18.900000,0 days 00:00:34.307000,0 days 00:00:19.221000,0 days 01:03:02.347000,0 days 01:03:36.654000,0 days 01:03:55.875000,220.0,203.0,270.0,280.0,True,SOFT,2.0,True,AlphaTauri,0 days 01:02:43.447000,2023-05-27 14:49:51.567,1,NaN,False,,False,True,2023,Monaco Grand Prix,1,0 days 00:01:12.428000,72.428,1970-01-01 00:00:08.321,00:00:08.321000


In [124]:
track_fastest.Year.unique()

<IntegerArray>
[2023, 2021, 2022, 2024, 2025]
Length: 5, dtype: Int64

In [125]:
track_fastest=track_fastest[["Driver","Year","GP","LapTime (s)"]].sort_values(by=["Year","GP","LapTime (s)"])
# track_fastest=track_fastest[~track_fastest["GP"].isin(["Pre-Season Test","Pre-Season Testing","Pre-Season Track Session","Emilia Romagna Grand Prix","French Grand Prix","Saudi Arabian Grand Prix"])]
# track_fastest=track_fastest[~((track_fastest["GP"].isin(["United States Grand Prix"]))&(track_fastest["Year"].isin([2024])))]
track_fastest_tsu=track_fastest[track_fastest["Driver"]=="TSU"].reset_index(drop=True).fillna(0)
track_fastest_tsu.columns=["Driver1","Year1","GP1","TSU_LapTime"]
track_fastest_driver2=track_fastest[track_fastest["Driver"]!="TSU"].reset_index(drop=True).fillna(0)
track_fastest_driver2.columns=["Driver2","Year2","GP2","D2_LapTime"]


In [126]:
track_final=track_fastest_tsu.merge(track_fastest_driver2, left_on=["GP1","Year1"], right_on=["GP2","Year2"])
track_final["Delta"]=track_final["TSU_LapTime"]-track_final["D2_LapTime"]
# track_final.to_csv("tsu.csv")
track_final.groupby(["Driver2"])["Delta"].mean()

Driver2
DEV    -1.549000
GAS   -13.097745
HAD    -0.173000
LAW    -0.643091
RIC    -0.199680
Name: Delta, dtype: float64

In [128]:
track_final.to_csv("tsu.csv")

In [12]:
fig=px.bar(
    track_fastest,
    y="Driver",
    x='LapTimeDelta',
    text='LapTimeDelta2',
    orientation='h',
    color='Driver',
    template="presentation",
    hover_data=['Team', 'Year', 'GP','LapTime', 'LapNumber','Sector1Time', 'Sector2Time', 'Sector3Time',
       'Compound', 'TyreLife', 'FreshTyre'],
    title="<b>Qualifying Delta vs. Fastest Lap for the {} {}</b>".format(year,gp),
    height=800, 
    width=1200,

color_discrete_map={
                 "VER": "#3671C6",
                 "LAW": "#3671C6",
                 "LEC": "#E80020",
                 "HAM": "#E80020",
                 "NOR": "#FF8000",
                 "PIA": "#FF8000",
                 "RUS": "#27F4D2",
                 "ANT": "#27F4D2",
                 "GAS": "#0093CC",
                 "DOO": "#0093CC",
                 "ALO": "#229971",
                 "STR": "#229971",
                 "SAI": "#64C4FF",
                 "ALB": "#64C4FF",
                 "HUL": "#52e252",
                 "BOR": "#52e252",
                 "TSU": "#6692FF",
                 "HAD": "#6692FF",
                 "OCO": "#B6BABD",
                 "BEA": "#B6BABD"
                 }
)
fig.update_layout(
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),

    xaxis_title="<b>Delta</b>",
    yaxis_title="<b>Driver</b>",
    title_font_family="<b>PT Sans Narrow</b>",

)

fig.update_layout(xaxis_tickformat='%H:%M:%S.%f')

fig.update_traces(marker_line_width=1,marker_line_color="BLACK")

fig.update_layout(yaxis={'categoryorder':'total descending'})

fig.update_traces(textposition='auto')

fig.update_layout(
    title_x=0.5,
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=20)),
    xaxis = dict(tickfont = dict(size=20)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow",
    margin=dict(l=60, r=5, t=35, b=60),
)
fig.show()